# 🚀 18TUBE Downloader - Tải Trực Tiếp Vào Google Drive Trên Google Colab
Hệ thống tự động tải Ảnh gốc & Video HD trực tiếp từ 18tube.nl vào Google Drive của bạn:
- ✅ **Không tốn dung lượng ổ cứng máy tính** (0 Byte).
- ✅ **Tốc độ mạng Google cực nhanh** (băng thông cáp quang Data Center).
- ✅ **Tích hợp sẵn Cloudscraper** vượt Cloudflare anti-bot trên Cloud.

In [ ]:
# @title 1. Kết nối tài khoản Google Drive của bạn 🔑
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Đã kết nối Google Drive thành công!")

In [ ]:
# @title 2. Cài đặt môi trường & Tải mã nguồn mới nhất ⚙️
import os, shutil

# Đưa thư mục làm việc về /content an toàn trước khi thao tác
try:
    os.chdir('/content')
except Exception:
    pass

# Cập nhật hoặc tải mới mã nguồn
if os.path.exists('/content/18tube/.git'):
    !git -C /content/18tube pull
else:
    !rm -rf /content/18tube
    !git clone https://github.com/dongt2/18tube-web.git /content/18tube

os.chdir('/content/18tube')
# Cài đặt curl_cffi (giả lập Chrome TLS fingerprint để vượt Cloudflare) và các thư viện cần thiết
!pip install -q curl_cffi cloudscraper requests beautifulsoup4 tqdm
!which ffmpeg > /dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)
print('✅ Đã chuẩn bị môi trường thành công!')

In [ ]:
# @title 3. Cấu hình & Bắt đầu Tải vào Google Drive 🚀
import os

# Nhập link Creator cần tải (mỗi dòng 1 link)
urls_text = "https://18tube.nl/mihobaby/" # @param {type:"string"}
quality = "720p" # @param ["best", "720p", "480p", "360p", "lowest"]
only = "all" # @param ["all", "photos", "videos"]
photo_threads = 16 # @param {type:"slider", min:1, max:32, step:1}
video_threads = 6 # @param {type:"slider", min:1, max:12, step:1}
drive_folder = "/content/drive/MyDrive/18TUBE" # @param {type:"string"}

# [Tùy chọn] Đăng nhập tài khoản 18TUBE (nếu máy chủ Cloud bị yêu cầu đăng nhập):
username = "" # @param {type:"string"}
password = "" # @param {type:"string"}
cookie = "" # @param {type:"string"}

urls = [u.strip() for u in urls_text.strip().splitlines() if u.strip()]
if not urls:
    print("❌ Vui lòng nhập ít nhất một đường dẫn Creator!")
else:
    urls_file = "/content/urls.txt"
    with open(urls_file, "w", encoding="utf-8") as f:
        f.write("\n".join(urls))

    print(f"[*] Bắt đầu tải {len(urls)} Creator vào Google Drive: {drive_folder}")
    os.makedirs(drive_folder, exist_ok=True)

    extra_args = ""
    if username.strip() and password.strip():
        extra_args += f' --user "{username.strip()}" --password "{password.strip()}"'
    if cookie.strip():
        extra_args += f' --cookie "{cookie.strip()}"'

    cmd = (
        f"python downloader.py "
        f"--file '{urls_file}' "
        f"--out '{drive_folder}' "
        f"--quality '{quality}' "
        f"--only '{only}' "
        f"--photo-threads {photo_threads} "
        f"--video-threads {video_threads}"
        f"{extra_args}"
    )
    !{cmd}


In [ ]:
# @title 4. Kiểm tra tính đầy đủ (Completeness Check) 🔍
creator_url = "https://18tube.nl/hime_tsu/" # @param {type:"string"}
drive_folder = "/content/drive/MyDrive/18TUBE" # @param {type:"string"}

!python downloader.py --url "{creator_url}" --out "{drive_folder}" --check
